In [1]:
import math
import numpy as np
import torch
import os
from ultralytics.models.yolo.detect import DetectionTrainer

# ================= 1. 定义消融专用隐私引擎 (AGC + 固定噪声) =================
class PrivacyEngine_AGC_FixedNoise:
    def __init__(self, model, epsilon=1.0, strategy='uniform', epochs=30):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.total_epochs = epochs
        
        # 识别 Head
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def step(self, current_epoch):
        # Warm-up
        if current_epoch < 3: return

        # 1. ❌ 移除退火：固定 decay_factor = 1.0
        # 这意味着噪声全程保持最大值，不随时间衰减
        decay_factor = 1.0 

        # 2. ✅ 保留 AGC (自适应梯度裁剪)
        grad_norms = []
        for p in self.model.parameters():
            if p.requires_grad and p.grad is not None:
                grad_norms.append(p.grad.norm(2).item())
        
        if not grad_norms: return
        
        # AGC 核心逻辑
        current_median = np.median(grad_norms)
        # 动态计算裁剪阈值
        clip_val = max(0.01, min(current_median * 1.5, 10.0))

        # 调试打印
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🧪 [Ablation DEBUG] Epoch {current_epoch} | AGC Clip: {clip_val:.4f} | Fixed Noise: {decay_factor:.1f}")
            self._logged_this_epoch = current_epoch

        # 3. 策略计算
        current_device = next(self.model.parameters()).device
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            factors.append(1.0) # Uniform 策略

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            
            # 应用 AGC 动态阈值
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor # 这里 decay_factor 恒为 1.0
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 定义消融训练器 =================
class Trainer_StrictAblation_AGC(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 调用 PrivacyEngine_AGC_FixedNoise
        self.privacy_engine = PrivacyEngine_AGC_FixedNoise(
            model, epsilon=1.0, strategy='uniform', epochs=30
        )
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行脚本 (Experiment 2: AGC Only) =================
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始严格消融实验 (AGC Only)")
print("📝 配置: GC10迁移 + AGC(动态C) + 固定噪声(无退火)")
print("🎯 目的: 验证 AGC 对稳定性的贡献")

if os.path.exists(PRETRAINED_GC10):
    trainer_ablation_agc = Trainer_StrictAblation_AGC(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_ablation',
        'name': 'exp1_ablation_agc_fixed_noise',
        'device': '0',
        'exist_ok': True,
        'freeze': 0 
    })
    trainer_ablation_agc.train()
else:
    print(f"❌ 找不到预训练权重: {PRETRAINED_GC10}")

🚀 开始严格消融实验 (AGC Only)
📝 配置: GC10迁移 + AGC(动态C) + 固定噪声(无退火)
🎯 目的: 验证 AGC 对稳定性的贡献
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.937, mosaic=1

In [2]:
import math
import numpy as np
import torch
import os
from ultralytics.models.yolo.detect import DetectionTrainer

# ================= 1. 定义消融专用隐私引擎 (固定裁剪 + 噪声退火) =================
class PrivacyEngine_FixedClip_Annealing:
    def __init__(self, model, epsilon=1.0, strategy='uniform', epochs=30, fixed_clip=1.0):
        self.model = model
        self.epsilon = epsilon
        self.strategy = strategy
        self.total_epochs = epochs
        self.fixed_clip = fixed_clip  # 🔒 新增：固定裁剪阈值
        
        # 识别 Head (保持不变)
        self.head_indices = self._find_head_indices()
        self.backbone_indices = list(range(10))

    def _find_head_indices(self):
        indices = []
        for name, module in self.model.named_modules():
            if hasattr(module, '__class__') and 'Detect' in str(module.__class__):
                try:
                    idx = int(name.split('.')[1])
                    indices.append(idx)
                except: pass
        if not indices: indices = [23]
        return indices

    def _get_noise_multiplier(self, current_epoch):
        # ✅ 保留：噪声退火逻辑
        min_decay = 0.1 
        progress = current_epoch / self.total_epochs
        cosine_decay = 0.5 * (1 + math.cos(math.pi * progress))
        decay_factor = min_decay + (1 - min_decay) * cosine_decay
        return decay_factor

    def step(self, current_epoch):
        # Warm-up
        if current_epoch < 3: return

        # 1. 计算退火系数
        decay_factor = self._get_noise_multiplier(current_epoch)

        # 2. ❌ 移除 AGC (自适应计算)，改为 ✅ 固定裁剪
        # 原 AGC 代码: current_median = np.median(grad_norms); clip_val = ...
        clip_val = self.fixed_clip  # 强制使用固定阈值 (C=1.0)

        # 调试打印 (方便你观察梯度是否会爆炸)
        if not hasattr(self, '_logged_this_epoch') or self._logged_this_epoch != current_epoch:
            print(f"🧪 [Ablation DEBUG] Epoch {current_epoch} | Fixed Clip: {clip_val:.1f} | Decay: {decay_factor:.4f}")
            self._logged_this_epoch = current_epoch

        # 3. 策略计算 (保持 Uniform)
        current_device = next(self.model.parameters()).device
        param_list = [p for p in self.model.parameters() if p.requires_grad and p.grad is not None]
        names_list = [n for n, p in self.model.named_parameters() if p.requires_grad and p.grad is not None]
        
        factors = []
        for n in names_list:
            factors.append(1.0) # Uniform 策略

        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_list:
            layer_eps = self.epsilon * weights[idx]
            
            # 执行裁剪
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            # 执行加噪
            c = np.sqrt(2 * np.log(1.25 / 1e-5))
            base_sigma = c * clip_val / (layer_eps + 1e-8)
            final_sigma = base_sigma * decay_factor # ✅ 应用退火
            
            p.grad.add_(torch.randn_like(p.grad) * final_sigma)
            idx += 1

# ================= 2. 定义消融训练器 =================
class Trainer_StrictAblation(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # ⚠️ 这里调用新的 PrivacyEngine_FixedClip_Annealing
        # ⚠️ fixed_clip=1.0 是标准 DP 的常用值，如果梯度爆炸，说明 AGC 是必须的
        self.privacy_engine = PrivacyEngine_FixedClip_Annealing(
            model, epsilon=1.0, strategy='uniform', epochs=30, fixed_clip=1.0
        )
        return model
    
    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0) # 这是一个兜底，防止数值溢出
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行脚本 (Experiment 3: Missing Link) =================
# 配置路径 (请根据实际情况调整)
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
PRETRAINED_GC10 = "/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt"

print("🚀 开始严格消融实验 (Missing Link)")
print("📝 配置: GC10迁移 + 固定裁剪(C=1.0) + 噪声退火")
print("🎯 目的: 验证单纯的噪声退火是否足够，还是必须配合AGC？")

if os.path.exists(PRETRAINED_GC10):
    trainer_ablation = Trainer_StrictAblation(overrides={
        'model': PRETRAINED_GC10,
        'data': FULL_YAML,
        'epochs': 30,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_ablation',
        'name': 'exp2_ablation_fixed_clip_annealing', # 实验名称
        'device': '0',
        'exist_ok': True,
        'freeze': 0  # 全量微调
    })
    trainer_ablation.train()
else:
    print(f"❌ 找不到预训练权重: {PRETRAINED_GC10}")

🚀 开始严格消融实验 (Missing Link)
📝 配置: GC10迁移 + 固定裁剪(C=1.0) + 噪声退火
🎯 目的: 验证单纯的噪声退火是否足够，还是必须配合AGC？
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/13_GC10_Pretrain_Base/weights/best.pt, momentum=0.9